# 03. Scope

**Tier:** Intermediate. **Why you'd use this:** You need a family of counters, each starting at a different number. One outer function should build an inner function that remembers its own start.

**After this lesson you can:** return an inner function that still sees an outer name; explain LEGB; catch a loop of lambdas that all see the last value.
About 25 minutes. **Uses:** [Arguments](02-arguments-lesson.ipynb)


## Quick review

1. Why does a default list get longer on every call that omits it?
2. What is the difference between mutating a parameter and rebinding it?

<details><summary>Answers</summary>

The default object is created once at `def` time. Mutation changes the shared object. Rebinding only changes the parameter slot.
</details>


## The idea

A name assigned in a function is local to that function. Reading a name walks LEGB: Local, Enclosing function, Global, Built-in. An inner function that reads an enclosing name keeps that link after the outer call returns. That is a closure. It captures the variable, not a snapshot of the value at creation time.

<details><summary>Explain more</summary>

`global` and `nonlocal` are how you assign into an outer scope on purpose. Prefer returning a value. A class body is not an enclosing scope for a comprehension or a nested function the way a `def` is.
</details>


## See it work

Build a counter that starts at a chosen number.


In [1]:
def make_counter(start):
    # 1. count lives in the enclosing function.
    count = start

    def tick():
        nonlocal count
        # 2. Each call reads and updates that same count.
        count = count + 1
        return count

    return tick


ones = make_counter(0)
tens = make_counter(10)
print(ones(), ones())
print(tens(), ones())


1 2
11 3


## Check yourself

Answer these before you run the cell.

1. What does `adder(2)(3)` return?
2. What do the three inner functions return, and why are they alike?


In [2]:
def adder(n):
    def add(x):
        return x + n

    return add


print(adder(2)(3))


def labels_wrong():
    built = []
    for i in range(3):

        def read():
            return i

        built.append(read)
    return built


first, second, third = labels_wrong()
print(first(), second(), third())


5
2 2 2


<details><summary>Answers</summary>

`adder(2)(3)` is `5`. The three inner functions all return `2`. They each close over the name `i`, and by the time they run, `i` is the last value from the loop.
</details>


## Why it works this way

Lookup walks LEGB: Local, Enclosing function, Global, Built-in. Assignment in a function makes a local unless you declare `nonlocal` or `global`. A closure stores a cell that points at the enclosing variable. Reading the cell happens when the inner function runs, not when it was defined. That is why a loop that builds `def read(): return i` sees the final `i` in every function. Each one held the variable. None of them copied the integer from that iteration. Deep dive: [Scopes and closures](../../internals/06-scopes/01-closures-lesson.ipynb).

`make_counter` keeps a separate `count` cell per outer call. `ones` and `tens` do not share a counter. The factory fix for the loop bug works the same way: `factory(i)` creates a new enclosing `n` for each iteration, so each `read` has its own cell. The cost of a closure is that cell and the inner function object. The design lets you build small functions that remember configuration without a class. The late-binding cost shows up only when the read happens after the enclosing name has moved on. Until then, the shared cell looks fine.

`__closure__` exposes the cell for teaching. `cell_contents` is the current value of that enclosing name. After `adder(2)`, the cell holds `2`. After a loop of broken handlers, every function's cell holds the same final `i`. That print is the proof that the bug is shared state, not three separate integers that somehow became equal by chance after the loop ended.


In [3]:
def adder(n):
    def add(x):
        return x + n

    return add


add_two = adder(2)
print(add_two.__closure__[0].cell_contents)
print(add_two(5))


2
7


## When to use it

Use a closure when a small function must remember configuration. Use a class when that state has several methods or needs a clear name. Don't fix late binding with a default argument until you know the loop bug. Prefer a factory that takes `n` as a parameter, like `adder(n)`, so each function has its own cell.


## Common mistake

Building handlers in a loop that all read the loop variable. Every handler sees the last value. The silent wrong answers are all `2`.


In [4]:
def labels():
    built = []
    for i in range(3):

        def read():
            return i

        built.append(read)
    return built


first, second, third = labels()
print(first(), second(), third())


2 2 2


Give each handler its own enclosing name. A small factory takes `i` as a parameter, so each returned function closes over a different cell.


In [5]:
def labels():
    built = []
    for i in range(3):

        def factory(n):
            def read():
                return n

            return read

        built.append(factory(i))
    return built


first, second, third = labels()
print(first(), second(), third())


0 1 2


## Try it

Change `make_counter` so `tick` takes a step size with a default of `1`. Why is that default safe when a default list was not?


## Recap

- LEGB is the lookup order. Assignment makes a local unless you say otherwise.
- A closure reads the enclosing variable when it runs.
- Loop handlers share one loop variable unless each gets its own cell.

**Next:** [practice](03-scope-practice.ipynb)
